In [1]:
import os, sys
project_root = os.path.abspath('..').replace('\\', '/')
if project_root not in [p.replace('\\', '/') for p in sys.path]:
    sys.path.append(project_root)

# 02 编码器模块 (core.encoders)\n\n提供各类特征编码功能，所有编码器均遵循 sklearn Transformer 接口规范。\n\n**支持的编码器：**\n- `WOEEncoder`: 证据权重编码（风控专用）\n- `TargetEncoder`: 目标编码\n- `CountEncoder`: 计数编码\n- `OneHotEncoder`: 独热编码\n- `OrdinalEncoder`: 序数编码\n- `QuantileEncoder`: 分位数编码\n- `CatBoostEncoder`: CatBoost编码\n- `GBMEncoder`: 梯度提升树编码器\n\n**数据说明**: 基于 `hscredit_yyp.xlsx`，目标变量为 `MOB1 > 3`

In [2]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from hscredit import init_setting
from hscredit.core.encoders import (
    WOEEncoder, TargetEncoder, CountEncoder, OneHotEncoder
)

init_setting()

# 加载数据
_roots = [Path.cwd(), Path.cwd() / 'examples', Path.cwd().parent]
_fp = None
for _r in _roots:
    for _n in ('hscredit_yyp.xlsx', 'hengshucredit_yyp.xlsx'):
        if (_r / _n).is_file():
            _fp = _r / _n
            break
    if _fp is not None:
        break
if _fp is None:
    raise FileNotFoundError('请将 hscredit_yyp.xlsx 放在 examples/')

df = pd.read_excel(_fp)

# 构造目标变量
df['target'] = (df['MOB1'] > 3).astype(int)

# 重命名为英文列名
df = df.rename(columns={
    '中智小牛分C3': 'score_c3',
    '珊瑚92': 'score_coral',
    '极光欺诈分6v1': 'score_fraud',
})

print(f"样本数: {len(df):,}")
print(f"坏样本率: {df['target'].mean():.2%}")

样本数: 970
坏样本率: 16.70%


## 1. 先进行分箱，再WOE编码

In [3]:
from hscredit.core.binning import OptimalBinning

# 对数值特征进行分箱
features = ['score_c3', 'score_coral']
binner = OptimalBinning(method='quantile', max_n_bins=5, min_bin_size=0.05)
binner.fit(df[features], df['target'])
df_binned = binner.transform(df[features])
df_binned.columns = [f'{c}_bin' for c in df_binned.columns]

print(f"分箱后的列: {df_binned.columns.tolist()}")
print("\n分箱结果预览:")
display(df_binned.head())

分箱后的列: ['score_c3_bin', 'score_coral_bin']

分箱结果预览:


,score_c3_bin,score_coral_bin
0,-1,-1
1,-1,-1
2,-1,-1
3,-1,-1
4,-1,-1


In [4]:
# WOE编码 - 逐个特征进行
for col in df_binned.columns:
    woe_encoder = WOEEncoder(cols=[col])
    woe_encoder.fit(df_binned[[col]], df['target'])
    df_binned[f'{col}_woe'] = woe_encoder.transform(df_binned[[col]])

print("WOE编码结果预览:")
woe_cols = [c for c in df_binned.columns if c.endswith('_woe')]
display(df_binned[woe_cols].head())

WOE编码结果预览:


,score_c3_bin_woe,score_coral_bin_woe
0,-0.0531,-0.0670
1,-0.0531,-0.0670
2,-0.0531,-0.0670
3,-0.0531,-0.0670
4,-0.0531,-0.0670


## 2. Target编码（目标均值编码）

In [5]:
# 使用原始数值特征进行Target编码
target_encoder = TargetEncoder(
    cols=['score_c3'],
    smoothing=10
)
target_encoder.fit(df[['score_c3']], df['target'])
df['score_c3_target'] = target_encoder.transform(df[['score_c3']])

print("Target编码结果预览:")
display(df[['score_c3', 'score_c3_target']].head(10))

Target编码结果预览:


,score_c3,score_c3_target
0,NaN,0.1670
1,NaN,0.1670
2,NaN,0.1670
3,NaN,0.1670
4,NaN,0.1670
5,NaN,0.1670
6,NaN,0.1670
7,NaN,0.1670
8,NaN,0.1670
9,NaN,0.1670


## 3. Count编码（计数编码）

In [6]:
# Count编码
count_encoder = CountEncoder(
    cols=['score_c3_bin'],
    normalize=False
)
count_encoder.fit(df_binned[['score_c3_bin']])
df_binned['score_c3_bin_count'] = count_encoder.transform(df_binned[['score_c3_bin']])

print("Count编码结果预览:")
display(df_binned[['score_c3_bin', 'score_c3_bin_count']].head(10))

Count编码结果预览:


,score_c3_bin,score_c3_bin_count
0,-1,663
1,-1,663
2,-1,663
3,-1,663
4,-1,663
5,-1,663
6,-1,663
7,-1,663
8,-1,663
9,-1,663


## 4. Pipeline中使用编码器

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from hscredit.core.models import LogisticRegression

# 准备数据
feature_cols = ['score_c3', 'score_coral']
X = df[feature_cols].fillna(df[feature_cols].median())
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 使用TargetEncoder + LR的Pipeline
pipeline = Pipeline([
    ('target_encoder', TargetEncoder(cols=feature_cols)),
    ('classifier', LogisticRegression())
])

# 训练
pipeline.fit(X_train, y_train)

# 预测
y_prob = pipeline.predict_proba(X_test)[:, 1]

# 评估
from hscredit.core.metrics import ks, auc
print(f"测试集KS: {ks(y_test, y_prob):.4f}")
print(f"测试集AUC: {auc(y_test, y_prob):.4f}")

测试集KS: 0.1322
测试集AUC: 0.5537
